## RTL-SDR: Center-Frequenz und Gain per Slider

← [RTL-SDR-Samples](rtl_sdr_samples.ipynb) | [Interaktiver Spektralplot](rtl_sdr_spectrum_interactive.ipynb)

Gleicher Inhalt wie die anderen RTL-SDR-Notebooks, aber mit **interaktiven Slidern** für **Mittenfrequenz (Center Frequency)** und **Gain**. Beim Ändern der Slider werden neue Samples mit den gewählten Einstellungen eingelesen und I/Q sowie Spektrum neu geplottet.

**Voraussetzung:** RTL-SDR angeschlossen, Treiber-Zelle ausgeführt. Optional: `pip install ipympl` für Zoom/Pan im Plot.

### Windows: Treiber-Pfad setzen (vor dem Import)

In [1]:
import os
from pathlib import Path

_driver_dir = None
for p in [Path.cwd()] + list(Path.cwd().parents):
    candidate = p / "rtl-sdr-driver"
    if candidate.exists() and (candidate / "librtlsdr.dll").exists():
        _driver_dir = candidate
        break
if _driver_dir is not None:
    _path = str(_driver_dir)
    os.environ["PATH"] = _path + os.pathsep + os.environ.get("PATH", "")
    if hasattr(os, "add_dll_directory"):
        os.add_dll_directory(_path)
    print("RTL-SDR Treiber gefunden:", _driver_dir)
else:
    print("Hinweis: rtl-sdr-driver (librtlsdr.dll) nicht gefunden.")

RTL-SDR Treiber gefunden: C:\_Git\KT-workspace\rtl-sdr-driver


### Slider: Center Frequency (MHz) und Gain

Nach Ändern eines Sliders: Gerät wird mit den neuen Werten geöffnet, Samples gelesen, Gerät geschlossen, Plot aktualisiert. **Nur ein Prozess** kann das RTL-SDR gleichzeitig nutzen.

In [2]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, VBox, HBox, Label, Output, Button
from IPython.display import display, clear_output

num_samples = 4096
sample_rate_hz = 2.048e6

slider_freq = FloatSlider(
    value=93.0,
    min=88.0,
    max=108.0,
    step=0.15,
    description='Center (MHz):',
    continuous_update=False,
    readout=True,
    readout_format='.1f',
)
slider_gain = FloatSlider(
    value=20.0,
    min=0.0,
    max=49.6,
    step=0.1,
    description='Gain (dB):',
    continuous_update=False,
    readout=True,
    readout_format='.1f',
)
btn_refresh = Button(description='REFRESH', tooltip='Neue Samples mit aktuellen Einstellungen einlesen')
out = Output()
_last_fig = None

def update_plot(change=None):
    global _last_fig
    center_freq_mhz = slider_freq.value
    gain_db = slider_gain.value
    with out:
        clear_output(wait=True)
        if _last_fig is not None:
            plt.close(_last_fig)
            _last_fig = None
        try:
            from rtlsdr import RtlSdr
            sdr = RtlSdr()
            sdr.sample_rate = sample_rate_hz
            sdr.center_freq = center_freq_mhz * 1e6
            sdr.gain = gain_db
            samples = sdr.read_samples(num_samples)
            sdr.close()
        except Exception as e:
            print("RTL-SDR Fehler:", e)
            return
        I = np.real(samples)
        Q = np.imag(samples)
        I_int = np.clip(np.round(I * 127), -128, 127).astype(np.int32)
        Q_int = np.clip(np.round(Q * 127), -128, 127).astype(np.int32)
        N = len(samples)
        fs = sample_rate_hz
        fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=False)
        _last_fig = fig
        axes[0].plot(I_int[:], color='C0', linewidth=0.6, drawstyle='steps-mid')
        axes[0].set_ylabel('I (raw int)')
        axes[0].set_title('I-Samples (Realteil)')
        axes[0].grid(True, alpha=0.3)
        axes[1].plot(Q_int[:], color='C1', linewidth=0.6, drawstyle='steps-mid')
        axes[1].set_ylabel('Q (raw int)')
        axes[1].set_title('Q-Samples (Imaginärteil)')
        axes[1].grid(True, alpha=0.3)
        X = np.fft.fft(samples)
        freq = np.fft.fftfreq(N, 1 / fs)
        magnitude = np.abs(X)
        freq_shifted = np.fft.fftshift(freq)
        magnitude_shifted = np.fft.fftshift(magnitude)
        step = max(1, N // 400)
        axes[2].plot(freq_shifted / 1e6, magnitude_shifted, color='C2', linewidth=0.6,
                     marker='o', markersize=2, markevery=step)
        axes[2].set_xlabel('Frequenz (MHz)')
        axes[2].set_ylabel('|FFT|')
        axes[2].set_title(f'Zweiseitiges Spektrum @ {center_freq_mhz:.1f} MHz, Gain {gain_db:.1f} dB')
        axes[2].grid(True, alpha=0.3)
        magnitude_dB = 20 * np.log10(magnitude_shifted + 1e-12)
        axes[3].plot(freq_shifted / 1e6, magnitude_dB, color='C3', linewidth=0.6,
                     marker='s', markersize=1.5, markevery=step)
        axes[3].set_xlabel('Frequenz (MHz)')
        axes[3].set_ylabel('|FFT| (dB)')
        axes[3].set_title('Zweiseitiges Spektrum (dB)')
        axes[3].grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

slider_freq.observe(update_plot, names='value')
slider_gain.observe(update_plot, names='value')
btn_refresh.on_click(lambda _: update_plot())
display(VBox([Label('Mittenfrequenz und Gain einstellen:'), HBox([slider_freq, slider_gain, btn_refresh]), out]))
update_plot()

→ [RTL-SDR-Samples](rtl_sdr_samples.ipynb) | [Interaktiver Spektralplot](rtl_sdr_spectrum_interactive.ipynb)